# Week 8 — Multi-Agent Price Predictor

Week 8 builds a full agent framework deployed on Modal.com with fine-tuned models.
This is a simplified local version using the same agent architecture:

- **SpecialistAgent** — few-shot expert (simulates the fine-tuned model)
- **FrontierAgent** — GPT-4o-mini with chain-of-thought reasoning
- **EnsembleAgent** — combines both estimates with a confidence-weighted average
- **Gradio UI** — paste a product, see each agent's estimate and the final answer

In [ ]:
import re
import random
import logging
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from datasets import load_dataset
from tqdm import tqdm

load_dotenv(override=True)
client = OpenAI()
MODEL = 'gpt-4o-mini'

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(name)s] %(message)s')

In [ ]:
# Load a small pool of Amazon items to use as few-shot examples

dataset = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_meta_Appliances',
    split='full',
    trust_remote_code=True
)

def clean(item):
    try:
        price = float(item['price'])
    except (ValueError, TypeError):
        return None
    if not (1 <= price <= 1000):
        return None
    title = item.get('title', '').strip()
    features = ' '.join(item.get('features', []))
    text = f"{title}. {features}".strip()
    if len(text) < 30:
        return None
    return {'text': text[:600], 'price': price}

items = [clean(d) for d in dataset]
items = [i for i in items if i]

random.seed(42)
random.shuffle(items)
example_pool = items[:300]
avg_price = sum(i['price'] for i in example_pool) / len(example_pool)

print(f'Example pool: {len(example_pool)} items | avg price: ${avg_price:.2f}')

In [ ]:
# --- Helper ---

def extract_price(text, fallback=avg_price):
    match = re.search(r'\$?([\d,]+\.?\d*)', text)
    if match:
        return float(match.group(1).replace(',', ''))
    return fallback

In [ ]:
# --- SpecialistAgent ---
# Simulates a fine-tuned model using few-shot examples from the training pool.
# The fine-tuned model in week 8 was trained on 800k items; we approximate
# this by showing the model 8 real price examples before asking.

class SpecialistAgent:
    NAME = 'SpecialistAgent'
    N_EXAMPLES = 8

    def __init__(self):
        self.log = logging.getLogger(self.NAME)

    def price(self, description):
        examples = random.sample(example_pool, self.N_EXAMPLES)
        messages = [{
            'role': 'system',
            'content': 'You are a product pricing specialist. Reply with ONLY a dollar amount. No explanation.'
        }]
        for ex in examples:
            messages.append({'role': 'user', 'content': f'Price this product:\n{ex["text"]}'})
            messages.append({'role': 'assistant', 'content': f'${ex["price"]:.2f}'})
        messages.append({'role': 'user', 'content': f'Price this product:\n{description}'})

        response = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=15)
        raw = response.choices[0].message.content.strip()
        result = extract_price(raw)
        self.log.info(f'Estimate: ${result:.2f} (raw: {raw!r})')
        return result


# Quick test
specialist = SpecialistAgent()
print(f'SpecialistAgent: ${specialist.price("Instant Pot Duo 7-in-1 Electric Pressure Cooker, 6 Quart"):.2f}')

In [ ]:
# --- FrontierAgent ---
# Uses GPT-4o-mini with chain-of-thought reasoning to estimate price.
# Mirrors the week 8 FrontierAgent which uses GPT-4o + RAG.

class FrontierAgent:
    NAME = 'FrontierAgent'

    def __init__(self):
        self.log = logging.getLogger(self.NAME)

    def price(self, description):
        prompt = f"""You are an expert at estimating retail prices of consumer products.

Think step by step:
1. What category is this product?
2. What are its key features or specs?
3. What is a typical price range for this category?
4. What is your best single price estimate?

End your response with: FINAL PRICE: $X.XX

Product: {description}"""

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=200,
        )
        raw = response.choices[0].message.content.strip()
        self.log.info(f'Reasoning:\n{raw}')

        # Extract from FINAL PRICE line first, fall back to any number
        final_match = re.search(r'FINAL PRICE:\s*\$?([\d,]+\.?\d*)', raw)
        result = float(final_match.group(1).replace(',', '')) if final_match else extract_price(raw)
        self.log.info(f'Estimate: ${result:.2f}')
        return result


# Quick test
frontier = FrontierAgent()
print(f'FrontierAgent: ${frontier.price("Instant Pot Duo 7-in-1 Electric Pressure Cooker, 6 Quart"):.2f}')

In [ ]:
# --- EnsembleAgent ---
# Runs both agents and combines their estimates.
# Uses a simple weighted average (FrontierAgent weighted slightly higher
# for its reasoning ability, SpecialistAgent for its domain examples).

class EnsembleAgent:
    NAME = 'EnsembleAgent'
    WEIGHTS = {'SpecialistAgent': 0.45, 'FrontierAgent': 0.55}

    def __init__(self):
        self.log = logging.getLogger(self.NAME)
        self.specialist = SpecialistAgent()
        self.frontier = FrontierAgent()

    def price(self, description):
        estimates = {
            'SpecialistAgent': self.specialist.price(description),
            'FrontierAgent':   self.frontier.price(description),
        }
        ensemble = sum(estimates[k] * self.WEIGHTS[k] for k in estimates)
        self.log.info(f'Estimates: {estimates} → Ensemble: ${ensemble:.2f}')
        return estimates, ensemble


# Quick test
ensemble = EnsembleAgent()
estimates, final = ensemble.price('Dyson V11 Cordless Vacuum Cleaner')
for name, val in estimates.items():
    print(f'  {name}: ${val:.2f}')
print(f'  Ensemble:       ${final:.2f}')

In [ ]:
# --- Gradio UI ---

agent = EnsembleAgent()

def predict(description):
    if not description.strip():
        return '', '', ''
    estimates, final = agent.price(description)
    return (
        f"${estimates['SpecialistAgent']:.2f}",
        f"${estimates['FrontierAgent']:.2f}",
        f"${final:.2f}",
    )

with gr.Blocks(title='Multi-Agent Pricer') as demo:
    gr.Markdown('## Multi-Agent Price Predictor')
    gr.Markdown('Enter a product description and see how each agent estimates the price.')

    desc = gr.Textbox(lines=4, label='Product Description',
                      placeholder='e.g. Dyson V11 Cordless Vacuum Cleaner with 60-min runtime...')
    btn  = gr.Button('Predict', variant='primary')

    with gr.Row():
        specialist_out = gr.Textbox(label='SpecialistAgent (few-shot)')
        frontier_out   = gr.Textbox(label='FrontierAgent (chain-of-thought)')
        ensemble_out   = gr.Textbox(label='Ensemble (final answer)')

    btn.click(fn=predict, inputs=desc, outputs=[specialist_out, frontier_out, ensemble_out])

demo.launch()